# Lab 09-05 · PN Junction Electrostatics

Companion to chapter [09-05](../../chapters/09-05-pn-junction-electrostatics.md). Handout: [README.md](README.md).

The junction is solved two ways: by integrating the depletion-approximation charge, and by solving the full nonlinear Poisson equation with Newton's method. Comparing them shows how good the approximation is.

In [1]:
import math
import numpy as np
from scipy.optimize import brentq
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
NI = 1e10
EPS = 11.7 * 8.8541878128e-14            # F/cm
print(f"eps_s = {EPS:.4e} F/cm")

eps_s = 1.0359e-12 F/cm


## Part 1 — The depletion approximation by numerical integration

Choose trial edges, enforce charge neutrality, integrate the charge for the field and the field for the potential, and adjust the width until the total potential equals $V_{bi} - V_A$.

In [2]:
def depletion(NA, ND, VA=0.0, n=4001):
    Vbi = VT * math.log(NA * ND / NI**2)
    def integrate(x, rho, E0, V0):          # trapezoidal integration of one uniformly charged side
        E = E0 + np.concatenate([[0], np.cumsum((rho[1:] + rho[:-1]) / 2 * np.diff(x))]) / EPS
        V = V0 + np.concatenate([[0], np.cumsum(-(E[1:] + E[:-1]) / 2 * np.diff(x))])
        return E, V
    def profile(W):
        xn, xp = W * NA / (NA + ND), W * ND / (NA + ND)
        x1, x2 = np.linspace(-xp, 0, n), np.linspace(0, xn, n)       # separate grids: x_p can be tiny
        E1, V1 = integrate(x1, np.full(n, -Q * NA), 0.0, 0.0)
        E2, V2 = integrate(x2, np.full(n, Q * ND), E1[-1], V1[-1])
        return np.concatenate([x1, x2]), np.concatenate([E1, E2]), np.concatenate([V1, V2])
    W = brentq(lambda W: profile(W)[2][-1] - (Vbi - VA), 1e-8, 1e-2)
    x, E, V = profile(W)
    return W, np.abs(E).max(), Vbi

W0, E0, Vbi = depletion(1e17, 1e16)
print(f"V_bi = {Vbi:.4f} V, W = {W0*1e4:.4f} um, peak field = {E0:.4e} V/cm")
assert abs(W0 * 1e4 - 0.3318) < 5e-4 and abs(E0 - 4.665e4) < 5e1

V_bi = 0.7738 V, W = 0.3318 um, peak field = 4.6648e+04 V/cm


## Part 2 — The full Poisson equation (the chapter's guided exercise)

Unknown: the electrostatic potential $\psi(x)$ measured from the Fermi level, with $n = n_ie^{q\psi/kT}$ and $p = n_ie^{-q\psi/kT}$. Newton's method on $\psi'' = -\rho/\varepsilon_s$.

**🔮 Predict:** how many Debye lengths wide is the n-side transition, from 10% to 90% of $N_D$? Is the exact peak field larger or smaller than the depletion approximation's?

In [3]:
def poisson(NA, ND, L=2e-4, n=16001):
    x = np.linspace(-L / 2, L / 2, n); h = x[1] - x[0]
    Nnet = np.where(x > 0, ND, -NA).astype(float)
    psi = np.where(x > 0, VT * np.log(ND / NI), -VT * np.log(NA / NI)).astype(float)
    for _ in range(100):
        nn, pp = NI * np.exp(psi / VT), NI * np.exp(-psi / VT)
        F = np.zeros(n)
        F[1:-1] = (psi[2:] - 2 * psi[1:-1] + psi[:-2]) / h**2 + Q * (pp - nn + Nnet)[1:-1] / EPS
        main = np.full(n, -2 / h**2) - Q / EPS * (pp + nn) / VT
        main[0] = main[-1] = 1
        lo = np.full(n - 1, 1 / h**2); up = lo.copy(); up[0] = 0; lo[-1] = 0
        d = spsolve(diags([lo, main, up], [-1, 0, 1], format="csc"), -F)
        psi += d
        if np.max(np.abs(d)) < 1e-12:
            break
    return x, psi, NI * np.exp(psi / VT), NI * np.exp(-psi / VT)

x, psi, nn, pp = poisson(1e17, 1e16)
right = x > 0
x10, x90 = np.interp([0.1e16, 0.9e16], nn[right], x[right])
LD = math.sqrt(EPS * VT / (Q * 1e16))
E_exact = np.abs(np.gradient(psi, x)).max()
xh_n = np.interp(0.5e16, nn[right], x[right]); xh_p = np.interp(0.5e17, pp[~right][::-1], x[~right][::-1])
print(f"transition 10-90%: {(x90-x10)*1e7:.1f} nm = {(x90-x10)/LD:.2f} Debye lengths")
print(f"peak field: exact {E_exact:.4e} V/cm vs depletion {E0:.4e} V/cm")
print(f"half-density width {(xh_n-xh_p)*1e4:.4f} um vs depletion {W0*1e4:.4f} um")
assert abs((xh_n - xh_p) * 1e4 - 0.3118) < 5e-4 and E_exact < E0

transition 10-90%: 140.9 nm = 3.45 Debye lengths
peak field: exact 4.5307e+04 V/cm vs depletion 4.6648e+04 V/cm
half-density width 0.3118 um vs depletion 0.3318 um


## Part 3 — Reverse bias and a simulated C–V measurement

Compute the depleted charge against bias, differentiate it for the capacitance, and fit $1/C^2$ against $V_A$ for a P⁺N junction.

In [4]:
NA, ND, A = 1e19, 5e15, 1e-4
Vbi_pn = VT * math.log(NA * ND / NI**2)
VA = np.linspace(-10, 0, 201)
charge = np.array([Q * ND * depletion(NA, ND, v)[0] * NA / (NA + ND) for v in VA]) * A
C = -np.gradient(charge, VA, edge_order=2)        # dQ/d|V|: charge grows as VA goes negative
slope, intercept = np.polyfit(VA, 1 / C**2, 1)
N_fit = -2 / (Q * EPS * A**2 * slope)
print(f"extracted doping {N_fit:.4e} cm^-3 (true N_A N_D/(N_A+N_D) = {NA*ND/(NA+ND):.4e}); V_bi from intercept {-intercept/slope:.4f} V")
assert abs(N_fit / (NA * ND / (NA + ND)) - 1) < 2e-3 and abs(-intercept / slope - Vbi_pn) < 5e-3

extracted doping 4.9973e+15 cm^-3 (true N_A N_D/(N_A+N_D) = 4.9975e+15); V_bi from intercept 0.8746 V


## Part 4 — A graded junction

Solve the self-consistent width and built-in potential of a linearly graded junction.

In [5]:
def graded(a, VA=0.0):
    Vbi_of = lambda W: 2 * VT * math.log(a * W / (2 * NI))
    W = brentq(lambda W: Vbi_of(W) - VA - Q * a * W**3 / (12 * EPS), 1e-8, 1e-3)
    return W, Vbi_of(W), Q * a * W**2 / (8 * EPS)
for a in (1e20, 1e22):
    W, Vb, E = graded(a)
    print(f"a = {a:.0e} cm^-4: W = {W*1e4:.4f} um, V_bi = {Vb:.4f} V, peak field = {E:.4e} V/cm")
assert abs(graded(1e20)[0] * 1e4 - 0.8029) < 5e-4

a = 1e+20 cm^-4: W = 0.8029 um, V_bi = 0.6671 V, peak field = 1.2463e+04 V/cm
a = 1e+22 cm^-4: W = 0.1860 um, V_bi = 0.8296 V, peak field = 6.6898e+04 V/cm


## Wrap-up

- Integrating the depletion charge reproduces every closed-form result of chapter §3.4.
- The exact solution has soft edges a few Debye lengths wide and a slightly lower peak field; the depletion approximation is accurate to a few percent at zero and reverse bias.
- A simulated C–V measurement recovers the doping of the lightly doped side from the slope of $1/C^2$.